# 🦀 Rust Interview MCQ — Part 6: Advanced Ownership & Memory (Q501–Q600)


### Q501. What happens when you partially move from a struct and then use the struct again?

```rust
struct Foo { a: String, b: i32 }
let f = Foo { a: "hi".into(), b: 42 };
let x = f.a;  // move
println!("{}", f.b);  // ?
```

- A) Compiles; only `f.a` was moved, `f.b` is still valid
- B) Does not compile; the entire `f` is considered partially moved and cannot be used
- C) Compiles; Rust automatically copies `f.b`
- D) Does not compile; you must use `std::mem::replace` to avoid the partial move

<details>
<summary>Show Answer</summary>

**Q501. B)** Rust treats a struct as a single unit for borrowing. After a partial move of `f.a`, the struct `f` is in a partially moved state. You cannot access any field of `f` (including `f.b`) because the struct is no longer fully initialized. The compiler rejects this.

</details>


### Q502. In a loop, you move a value in the first iteration. What happens?

```rust
let v = vec![1, 2, 3];
for i in 0..2 {
    let _ = v;  // move on first iteration
}
```

- A) Compiles; the move happens only once
- B) Does not compile; `v` is moved in the first iteration and unavailable in the second
- C) Compiles; Rust clones `v` for each iteration
- D) Does not compile; you cannot move in a loop at all

<details>
<summary>Show Answer</summary>

**Q502. B)** On the first iteration, `v` is moved into `_`. On the second iteration, `v` is used again but it has already been moved. The compiler detects this and reports an error: use of moved value.

</details>


### Q503. In a `match` expression, moving from an enum variant in one arm affects other arms. Which is valid?

- A) Moving from `Some(x)` in one arm and using `x` in another arm
- B) Moving from different fields of a struct in different match arms
- C) Moving from the same binding in all arms (each arm consumes it)
- D) Moving from `Ok(x)` in one arm and referencing `x` in the `Err` arm

<details>
<summary>Show Answer</summary>

**Q503. C)** In a match, each arm gets its own ownership of the matched value. So moving in each arm is fine—each arm consumes the value independently. A, B, and D describe scenarios where you'd use the same value twice or across arms, which is invalid.

</details>


### Q504. What does this code do?

```rust
let mut x = Some(String::from("hello"));
match x {
    Some(ref s) => println!("{}", s),
    None => {}
}
// Can we use x here?
```

- A) Does not compile; `ref` in a match arm borrows `x`, so `x` cannot be used after
- B) Compiles; `ref s` borrows the inner String, and `x` remains usable (still `Some(...)`)
- C) Does not compile; `ref` is deprecated
- D) Compiles; `ref s` moves the String out, so `x` becomes `None`

<details>
<summary>Show Answer</summary>

**Q504. B)** `ref s` creates a reference to the inner value without moving it. The `Option` remains intact. After the match, `x` is still `Some(String)` and can be used. No move occurs.

</details>


### Q505. Consider: `let (a, b) = (vec![1], vec![2]);` then `drop(a);` then `drop(b);`. What happens?

- A) Both drops compile; each variable owns its vector
- B) Does not compile; `drop(a)` moves `a`, so `drop(b)` would use a moved value
- C) `drop(a)` consumes `a`; `drop(b)` is fine since `b` is separate
- D) Both A and C are correct

<details>
<summary>Show Answer</summary>

**Q505. D)** `a` and `b` are separate bindings from the destructuring. `drop(a)` consumes `a` and is valid. `drop(b)` consumes `b` and is also valid. Each owns its own value. Both A and C correctly describe this.

</details>


### Q506. What is the result of `let x = (String::new(), 0); let y = x.0;`?

- A) `x` is fully moved; `x.1` is inaccessible
- B) `x` is partially moved; `x.0` is gone but `x.1` is still valid
- C) Does not compile; you cannot move from a tuple field
- D) `x` is copied; both `x.0` and `x.1` remain

<details>
<summary>Show Answer</summary>

**Q506. B)** Moving `x.0` (the String) leaves the tuple in a partially moved state. The tuple type is `(String, i32)`; after moving the String, `x.1` (the i32) is still there. However, you cannot access `x` as a whole or `x.1` through `x` in standard Rust—the struct/tuple is partially moved. Actually in Rust, for tuples, you CAN access `x.1` after moving `x.0`! Let me check: In Rust, for a tuple `(a, b)`, if you do `let y = x.0`, then `x.1` is still accessible. So B is correct—partial move, and the remaining field is valid. The nuance is that `x` as a whole is 'partially moved' but you can still use `x.1`.

</details>


### Q507. In a `for` loop over `vec.into_iter()`, when does the vector get dropped?

- A) After each iteration
- B) When the iterator is exhausted (after the loop ends)
- C) Before the first iteration
- D) Never; the iterator takes ownership and the vector is forgotten

<details>
<summary>Show Answer</summary>

**Q507. B)** `into_iter()` consumes the vector and produces an iterator that owns the elements. The vector's buffer is dropped when the iterator is dropped, which happens when the loop ends (iterator exhausted) or when the iterator goes out of scope.

</details>


### Q508. What happens with `let v = vec![1,2,3]; for x in v { } for x in v { }`?

- A) Compiles; the first loop borrows, the second consumes
- B) Does not compile; `v` is moved in the first `for` (which uses `into_iter()`)
- C) Compiles; `for` uses `iter()` by default
- D) Compiles; Rust implicitly clones `v` for the second loop

<details>
<summary>Show Answer</summary>

**Q508. B)** `for x in v` implicitly calls `v.into_iter()`, which consumes `v`. After the first loop, `v` is moved and no longer valid. The second loop would try to use `v` again, causing a compile error.

</details>


### Q509. `if let Some(x) = option` moves from `option`. How do you match without moving?

- A) `if let Some(ref x) = option`
- B) `if let Some(x) = &option`
- C) `if let Some(ref x) = &option`
- D) Both A and C work

<details>
<summary>Show Answer</summary>

**Q509. D)** `ref x` borrows the inner value. `&option` borrows the Option. `if let Some(ref x) = option` borrows the inner value but consumes the Option (moves it). `if let Some(ref x) = &option` borrows both—no move. For 'match without moving', C is the typical choice. A also avoids moving the inner value but moves the Option. So both can be used depending on whether you need the Option afterward.

</details>


### Q510. After `let (a, _) = (String::new(), 0);`, can you use `a`?

- A) No; `_` means the whole tuple is ignored
- B) Yes; `a` binds to the first element, `_` ignores the second
- C) No; destructuring with `_` consumes the whole tuple
- D) Yes; but only if you use `ref a`

<details>
<summary>Show Answer</summary>

**Q510. B)** `let (a, _) = ...` binds `a` to the first element and ignores the second with `_`. `a` is a valid binding and can be used. The `_` does not affect `a`.

</details>


### Q511. What is reborrowing?

- A) Converting `&mut T` to `&T` to allow multiple readers
- B) Creating a new reference from an existing reference, e.g. `&*r` or `&mut *r`
- C) Converting a shared reference to a mutable reference
- D) Moving a reference into a new binding

<details>
<summary>Show Answer</summary>

**Q511. B)** Reborrowing creates a new reference from an existing one. For `&mut T`, you can reborrow with `&mut *r` to get a shorter-lived mutable reference, allowing the original to be used again later. For `&T`, `&*r` reborrows.

</details>


### Q512. Why does `let r = &mut *v;` sometimes allow `v` to be used again later?

- A) It creates a copy of the reference
- B) The reborrow `&mut *v` has a shorter lifetime; when `r` goes out of scope, `v` is usable again
- C) It doesn't; `v` is always moved
- D) The compiler inserts a clone

<details>
<summary>Show Answer</summary>

**Q512. B)** `&mut *v` reborrows from `v`, creating a new mutable reference with a shorter lifetime. When `r` (the reborrow) goes out of scope, the borrow ends and `v` can be used again. This is key to NLL (Non-Lexical Lifetimes).

</details>


### Q513. What is borrow splitting?

- A) Splitting a `&mut [T]` into two non-overlapping `&mut [T]` slices
- B) Converting `&T` into `&mut T`
- C) Splitting a reference's lifetime in half
- D) Using `split_at_mut` to get two overlapping mutable slices

<details>
<summary>Show Answer</summary>

**Q513. A)** Borrow splitting allows multiple mutable borrows of different parts of the same data. `slice.split_at_mut(mid)` yields `(&mut self[..mid], &mut self[mid..])`—two non-overlapping mutable slices. The borrow checker recognizes they don't alias.

</details>


### Q514. What are two-phase borrows?

- A) A borrow that is first reserved, then activated when the reference is actually used
- B) A borrow that starts as shared and becomes mutable
- C) Two separate borrows of the same data
- D) A borrow that lasts for two function calls

<details>
<summary>Show Answer</summary>

**Q514. A)** Two-phase borrows allow `vec.push(vec.len())` to compile. The mutable borrow for `push` is first 'reserved' (not yet active). The `vec.len()` shared borrow runs, then the mutable borrow is 'activated' for the actual `push`. This enables method call patterns that would otherwise be rejected.

</details>


### Q515. NLL (Non-Lexical Lifetimes) allows:

- A) References to outlive their lexical scope
- B) A borrow to end as soon as the reference is last used, not at the end of the block
- C) Multiple mutable borrows of the same data
- D) References to be used after the borrow ends

<details>
<summary>Show Answer</summary>

**Q515. B)** NLL allows the borrow checker to end a borrow at the point of last use, rather than at the end of the block. This enables code like: use a reference, then use the original value in the same block, without extending the borrow unnecessarily.

</details>


### Q516. Why does this compile?

```rust
let mut v = vec![1, 2, 3];
let r = &mut v;
r.push(4);
let n = v.len();
```

- A) `r` is not used after `push`, so the borrow ends before `v.len()`
- B) `v.len()` takes a shared borrow, which is allowed with a mutable borrow
- C) It doesn't compile
- D) `r` and `v` are different names for the same data, so it's fine

<details>
<summary>Show Answer</summary>

**Q516. A)** With NLL, after `r.push(4)`, `r` is no longer used. The borrow of `v` ends at that point. So `v.len()` is valid—the mutable borrow has ended. This is Non-Lexical Lifetimes in action.

</details>


### Q517. What does `#![feature(nll)]` do in modern Rust?

- A) Enables the NLL borrow checker (default since Rust 2018)
- B) Enables a stricter, experimental borrow checker
- C) Disables the borrow checker
- D) Enables polymorphic NLL for higher-ranked lifetimes

<details>
<summary>Show Answer</summary>

**Q517. A)** NLL is enabled by default in Rust 2018 and later. The `#![feature(nll)]` was used in older editions to opt in. In modern Rust, NLL is always on; the feature flag is obsolete.

</details>


### Q518. In `fn foo(x: &mut i32) { let y = &mut *x; }`, what is the relationship between `x` and `y`?

- A) `y` is an independent copy of `x`
- B) `y` is a reborrow of `x`; both point to the same memory
- C) `y` takes ownership of `*x`
- D) `y` is a shared reference to `x`

<details>
<summary>Show Answer</summary>

**Q518. B)** `&mut *x` reborrows from `x`. `y` is a new mutable reference to the same `i32` that `x` points to. They alias the same memory; `y` has a shorter lifetime (the function body).

</details>


### Q519. `fn f(s: &str) -> &str` has an implicit lifetime. What does the compiler infer?

- A) The output lives as long as the input
- B) The output has a static lifetime
- C) The output has a fresh local lifetime
- D) The compiler cannot infer and requires an explicit lifetime

<details>
<summary>Show Answer</summary>

**Q519. A)** Lifetime elision rule: for `fn f(s: &str) -> &str`, the single input reference gets its lifetime applied to the output. So the return type is inferred as `&'a str` where `'a` is the lifetime of `s`.

</details>


### Q520. Can you have two mutable borrows of the same `Vec` at the same time?

- A) Yes, with `unsafe`
- B) Yes, using `split_at_mut` to get non-overlapping slices
- C) No, never
- D) Yes, with `RefCell`

<details>
<summary>Show Answer</summary>

**Q520. B)** `split_at_mut` splits a slice into two non-overlapping mutable parts. The borrow checker treats them as distinct borrows. So you can have `let (a, b) = v.split_at_mut(2);` and use both `a` and `b` mutably. C and D are wrong; A is unnecessary for this pattern.

</details>


### Q521. For `fn f(a: &i32, b: &i32) -> &i32`, what does lifetime elision infer?

- A) All three references share the same lifetime
- B) The output lifetime is the longer of the two inputs
- C) The output lifetime is the shorter of the two inputs
- D) Compiler error; cannot elide with two inputs

<details>
<summary>Show Answer</summary>

**Q521. D)** With two input references, the compiler cannot infer which lifetime applies to the output. Elision rules don't cover this case. You must write explicitly: `fn f<'a>(a: &'a i32, b: &'a i32) -> &'a i32` or similar.

</details>


### Q522. What does `for<'a>` mean in `F: for<'a> Fn(&'a str) -> &'a str`?

- A) The function must work for one specific lifetime `'a`
- B) The function must work for any lifetime `'a` (higher-ranked trait bound)
- C) The function takes a reference with lifetime `'a`
- D) `'a` is a loop variable

<details>
<summary>Show Answer</summary>

**Q522. B)** `for<'a>` is a Higher-Ranked Trait Bound (HRTB). It means the type must implement the trait for every possible lifetime `'a`. The function must be callable with any `&'a str` and return `&'a str`.

</details>


### Q523. `fn foo(x: &i32, y: &i32)` — what are the elided lifetimes?

- A) Both get the same anonymous lifetime
- B) Each gets a distinct lifetime; no relationship
- C) `x` and `y` must outlive the function
- D) The compiler assigns `'static` to both

<details>
<summary>Show Answer</summary>

**Q523. B)** For multiple input references with no output reference, each gets its own distinct anonymous lifetime. There is no constraint that they be the same. The elision rules don't tie them together.

</details>


### Q524. What is the lifetime of the return value in `fn f() -> &str`?

- A) `'static` (elision rule for no inputs)
- B) Compiler error; cannot infer
- C) A fresh local lifetime
- D) The lifetime of the function

<details>
<summary>Show Answer</summary>

**Q524. B)** When there are no input references, the elision rules don't apply. Returning `&str` without any input reference means the compiler cannot infer the lifetime. You must write `-> &'static str` explicitly if that's intended.

</details>


### Q525. `trait Trait { fn f(&self) -> &str; }` — what lifetime does `&str` have?

- A) `'static`
- B) Tied to `&self`'s lifetime (elision: `fn f(&self) -> &str` → output = self)
- C) A new local lifetime
- D) Must be written explicitly

<details>
<summary>Show Answer</summary>

**Q525. B)** In trait methods, the elision rule for `fn f(&self) -> &str` assigns the output lifetime to `&self`. So the returned `&str` lives as long as `self`.

</details>


### Q526. `where for<'a> T: Fn(&'a str)` — what does this constrain?

- A) `T` must implement `Fn` for one specific `'a`
- B) `T` must implement `Fn(&'a str)` for every possible lifetime `'a`
- C) `T` has a lifetime parameter `'a`
- D) `T` outlives `'a`

<details>
<summary>Show Answer</summary>

**Q526. B)** HRTB: `for<'a> T: Fn(&'a str)` means `T` must be callable with `&str` for any lifetime. This is needed when the closure/function is generic over the lifetime of its argument.

</details>


### Q527. Why might you need `impl for<'a> Fn(&'a str) -> &'a str` for a closure?

- A) Closures don't have lifetime parameters
- B) The closure must work when given a reference with any lifetime
- C) To allow the closure to outlive its capture
- D) To satisfy the `Fn` trait's requirements

<details>
<summary>Show Answer</summary>

**Q527. B)** A closure that returns a reference to its argument (e.g. `|s| s`) is generic over the input lifetime. The type system needs `for<'a> Fn(&'a str) -> &'a str` to express that it works for any `'a`.

</details>


### Q528. `fn f<'a: 'b, 'b>(x: &'a i32, y: &'b i32)` — what does `'a: 'b` mean?

- A) `'a` is shorter than `'b`
- B) `'a` outlives `'b` (i.e. `'a` is at least as long as `'b`)
- C) `'b` outlives `'a`
- D) `'a` and `'b` are the same

<details>
<summary>Show Answer</summary>

**Q528. B)** `'a: 'b` is a lifetime bound: `'a` outlives `'b`. Any reference with lifetime `'a` lives at least as long as one with `'b`. It's read as 'a outlives b'.

</details>


### Q529. What does `T: 'static` mean?

- A) `T` contains no references
- B) `T` contains no non-`'static` references (all refs live forever or there are no refs)
- C) `T` is a static variable
- D) `T` has a `'static` lifetime parameter

<details>
<summary>Show Answer</summary>

**Q529. B)** `T: 'static` means every reference in `T` (if any) has `'static` lifetime. Types like `i32`, `String`, `Vec<i32>` are `'static` because they own their data. `&'static str` is also `'static`.

</details>


### Q530. `fn f<T>(x: T) where T: 'static` — what can you pass?

- A) Only `&'static` references
- B) Any owned type (no references) or types with only `'static` refs
- C) Any type
- D) Only `String` and `Vec`

<details>
<summary>Show Answer</summary>

**Q530. B)** `T: 'static` accepts owned types (`i32`, `String`, `Vec<T>`) and types with only `'static` references. You cannot pass `&'a str` where `'a` is not `'static`.

</details>


### Q531. What is the primary use of `Box<T>`?

- A) To share ownership across threads
- B) To allocate `T` on the heap and have a single owner
- C) To allow interior mutability
- D) To create reference cycles

<details>
<summary>Show Answer</summary>

**Q531. B)** `Box<T>` allocates `T` on the heap. There is a single owner; when the `Box` is dropped, the heap allocation is freed. It's for heap allocation and indirection, not sharing or interior mutability.

</details>


### Q532. When would you use `Rc<T>` instead of `Box<T>`?

- A) When you need thread safety
- B) When multiple owners need to share the same value
- C) When you need mutable access from multiple places
- D) When the value is very large

<details>
<summary>Show Answer</summary>

**Q532. B)** `Rc<T>` provides shared ownership via reference counting. Multiple `Rc` clones point to the same allocation. Use it when several parts of the program need to own the value. For thread-safe sharing, use `Arc<T>`.

</details>


### Q533. What is `Weak<T>` used for?

- A) To create a weak reference that doesn't prevent dropping
- B) To allow mutable access without interior mutability
- C) To reduce the size of `Rc`
- D) To break reference cycles and avoid memory leaks

<details>
<summary>Show Answer</summary>

**Q533. D)** `Weak<T>` doesn't contribute to the strong count. When all `Rc`s are dropped, the value is dropped even if `Weak` references remain. `Weak::upgrade()` yields `Option<Rc<T>>`. This breaks cycles: A→Rc(B), B→Rc(A) would leak; B→Weak(A) allows both to be dropped.

</details>


### Q534. `Rc` vs `Arc`: when do you need `Arc`?

- A) Always; `Arc` is the modern replacement for `Rc`
- B) When the value is shared across threads
- C) When you need atomic operations on the value
- D) When the reference count exceeds 2^31

<details>
<summary>Show Answer</summary>

**Q534. B)** `Arc` uses atomic reference counting and is thread-safe. Use it when multiple threads need to share ownership. `Rc` is for single-threaded use only; using it across threads would be undefined behavior.

</details>


### Q535. How do you create a reference cycle with `Rc`?

```rust
use std::rc::Rc;
struct Node { next: Option<Rc<Node>> }
let a = Rc::new(Node { next: None });
let b = Rc::new(Node { next: Some(a.clone()) });
// How to make a cycle?
```

- A) `a.next = Some(b.clone())` — but you can't mutate through `Rc`
- B) Use `Rc::get_mut` to mutate `a`
- C) Use `RefCell` (or similar) to allow `a` to hold `Rc` to `b`
- D) You cannot create a cycle with `Rc` alone

<details>
<summary>Show Answer</summary>

**Q535. C)** `Rc` is immutable. To create a cycle, you need interior mutability. `Rc<RefCell<Node>>` lets you mutate the `next` field so `a` can point to `b` while `b` points to `a`. That creates a cycle.

</details>


### Q536. What happens when you `clone()` an `Rc<T>`?

- A) Deep copy of `T`
- B) Increments the reference count; both clones point to the same allocation
- C) Moves the value to the new `Rc`
- D) Creates a weak reference

<details>
<summary>Show Answer</summary>

**Q536. B)** `Rc::clone` increments the strong reference count. The new `Rc` points to the same allocation. No copying of `T` occurs. Dropping an `Rc` decrements the count; when it reaches 0, `T` is dropped.

</details>


### Q537. `Box::leak(Box::new(x))` returns `&'static mut T`. What does it do?

- A) Leaks the box and returns a mutable reference that lives forever
- B) Creates a static variable
- C) Returns a reference that will be freed when it goes out of scope
- D) Causes undefined behavior

<details>
<summary>Show Answer</summary>

**Q537. A)** `Box::leak` consumes the `Box` and returns `&'static mut T`. The allocation is never freed (intentional leak). The reference is valid for the rest of the program. Used when you need a `'static` reference (e.g. for `lazy_static`-style initialization).

</details>


### Q538. Why might `Rc<RefCell<T>>` be used?

- A) For thread-safe shared mutable state
- B) For single-threaded shared mutable state (multiple owners, interior mutability)
- C) To avoid reference cycles
- D) To reduce allocation count

<details>
<summary>Show Answer</summary>

**Q538. B)** `Rc` gives shared ownership; `RefCell` gives interior mutability. Together they allow multiple owners to mutate the same value at runtime (with borrow checks at runtime). For threads, you'd use `Arc<Mutex<T>>` or similar.

</details>


### Q539. What is the strong count of a new `Rc::new(x)`?

- A) 0
- B) 1
- C) 2
- D) Depends on `x`

<details>
<summary>Show Answer</summary>

**Q539. B)** A newly created `Rc` has a strong count of 1. Each `clone()` increments it. Each `drop` decrements it. When it reaches 0, the inner value is dropped.

</details>


### Q540. `Arc<T>` requires `T: Send + Sync`. Why?

- A) `Arc` clones can move between threads; `T` may be accessed from multiple threads
- B) `Arc` is always sent across threads
- C) `T` must be `Copy`
- D) To allow `Arc` to be `Clone`

<details>
<summary>Show Answer</summary>

**Q540. A)** `Arc` is designed for shared ownership across threads. Cloning and sending between threads means `T` might be accessed from different threads. So `T: Send + Sync` is required for safe concurrent access.

</details>


### Q541. What does `Cell<T>` provide?

- A) Thread-safe interior mutability
- B) Interior mutability for `Copy` types (or types where you swap whole values)
- C) A shared reference that allows mutation
- D) Mutable static storage

<details>
<summary>Show Answer</summary>

**Q541. B)** `Cell<T>` provides interior mutability by allowing you to replace the whole value with `get()` (for `Copy`) or `set()`/`replace()`/`swap()`. It doesn't give you a `&mut T`; you work with owned values. Not thread-safe.

</details>


### Q542. `RefCell<T>` vs `Cell<T>`: when use `RefCell`?

- A) When `T` is `Copy`
- B) When you need to get `&T` or `&mut T` and mutate through a shared reference
- C) When you need thread safety
- D) When `T` is large

<details>
<summary>Show Answer</summary>

**Q542. B)** `RefCell` lets you obtain `&T` or `&mut T` via `borrow()`/`borrow_mut()` at runtime. It enforces the borrowing rules at runtime (panics on violation). Use it when you need to mutate through a shared reference and can't use `&mut`.

</details>


### Q543. What happens if you call `borrow_mut()` on a `RefCell` while a `borrow()` is active?

- A) Compiles but panics at runtime
- B) Compiles and runs; RefCell allows multiple mutable borrows
- C) Does not compile
- D) Undefined behavior

<details>
<summary>Show Answer</summary>

**Q543. A)** `RefCell` enforces borrowing at runtime. If you `borrow()` and then `borrow_mut()` (or vice versa) while the first is still active, it panics. Same rules as compile-time borrows, but checked at runtime.

</details>


### Q544. What is `UnsafeCell<T>`?

- A) A safe wrapper for interior mutability
- B) The primitive for interior mutability; `Cell` and `RefCell` are built on it
- C) A thread-safe `Cell`
- D) A type that allows arbitrary unsafe mutations

<details>
<summary>Show Answer</summary>

**Q544. B)** `UnsafeCell<T>` is the core primitive. It has `get() -> *mut T` and is the only way to get a mutable pointer from an immutable reference in safe Rust's model. `Cell` and `RefCell` wrap it to provide safe APIs.

</details>


### Q545. What is `OnceCell<T>` (or `std::sync::OnceLock<T>`)?

- A) A cell that can be set only once and then read many times
- B) A cell that can be set multiple times
- C) A thread-safe `Cell`
- D) A lazy static

<details>
<summary>Show Answer</summary>

**Q545. A)** `OnceCell`/`OnceLock` holds a value that can be written at most once. After that, it's read-only. Used for lazy initialization: compute on first access, then cache. `OnceLock` is the thread-safe version in std.

</details>


### Q546. `OnceLock::get_or_init(|| expensive())` — when is `expensive()` called?

- A) Every time `get_or_init` is called
- B) Only the first time; subsequent calls return the cached value
- C) When the `OnceLock` is created
- D) Never; it's lazy

<details>
<summary>Show Answer</summary>

**Q546. B)** `get_or_init` runs the closure only if the cell is empty. The first caller runs it and stores the result. Later callers get the cached value without running the closure again.

</details>


### Q547. Can `Cell<Vec<i32>>` be used to push to the vector through a shared reference?

- A) No; `Cell` doesn't give `&mut Vec`
- B) Yes; `Cell` allows any mutation
- C) Yes; with `get_mut()` 
- D) Only with `unsafe`

<details>
<summary>Show Answer</summary>

**Q547. A)** `Cell` provides `get()` (for `Copy`), `set()`, `replace()`, `swap()`. It doesn't give `&mut T`. To push to a `Vec` you need `&mut Vec`. So you'd use `cell.replace(cell.take().into_iter().chain([x]).collect())` or similar—or use `RefCell<Vec<i32>>` instead.

</details>


### Q548. Why is `RefCell` not `Sync`?

- A) It doesn't implement `Send`
- B) Sharing a `RefCell` across threads could allow concurrent `borrow` and `borrow_mut`, causing data races
- C) It's actually `Sync`
- D) It uses non-atomic operations

<details>
<summary>Show Answer</summary>

**Q548. B)** `RefCell` uses non-atomic ref counting for its borrow state. If shared across threads, two threads could `borrow` and `borrow_mut` concurrently, leading to data races. So it's not `Sync`. Use `Mutex` or `RwLock` for thread-safe interior mutability.

</details>


### Q549. `thread_local! { static X: RefCell<i32> = RefCell::new(0); }` — is this safe?

- A) No; RefCell is not thread-safe
- B) Yes; each thread has its own `X`, so no sharing
- C) No; thread_local is unsafe
- D) Only with `Sync`

<details>
<summary>Show Answer</summary>

**Q549. B)** `thread_local!` gives each thread its own instance of `X`. There's no sharing between threads, so `RefCell`'s non-`Sync` nature doesn't matter. Each thread mutates its own copy. Safe.

</details>


### Q550. What does `Cell::from_mut(&mut T)` do?

- A) Creates a `Cell` that borrows `T`
- B) Creates a `Cell` from a mutable reference, treating the referent as a `Cell` (for splitting)
- C) Moves `T` into a `Cell`
- D) Converts `&mut T` to `&Cell<T>`

<details>
<summary>Show Answer</summary>

**Q550. B)** `Cell::from_mut` takes `&mut T` and returns `&Cell<T>`. It's used with `Cell::from_mut(slice).as_slice_of_cells()` to get `&[Cell<T>]` from `&mut [T]`, enabling per-element interior mutability.

</details>


### Q551. What problem does `Pin` solve?

- A) Preventing values from being moved in memory
- B) Enabling self-referential structs (where a field points to another field)
- C) Making async types `Send`
- D) Preventing data races

<details>
<summary>Show Answer</summary>

**Q551. B)** Self-referential structs can have a pointer to their own data. Moving the struct would invalidate that pointer. `Pin` ensures the pointee is not moved, enabling safe self-referential types (e.g. async generators, futures).

</details>


### Q552. What does `Unpin` mean?

- A) The type cannot be pinned
- B) The type is safe to move even when pinned (pinning has no effect)
- C) The type has no self-references
- D) The type is always pinned

<details>
<summary>Show Answer</summary>

**Q552. B)** `Unpin` is an auto trait. Types that implement it (most types) can be moved out of a `Pin<P>`. For `Unpin` types, `Pin` is a no-op. Types with self-references typically don't implement `Unpin`.

</details>


### Q553. `Pin<Box<MyFuture>>` — what guarantee does this provide?

- A) The `Box` cannot be moved
- B) The future inside the `Box` cannot be moved in memory (stable address)
- C) The future cannot be polled
- D) The `Box` is on the stack

<details>
<summary>Show Answer</summary>

**Q553. B)** `Pin<Box<T>>` pins `T` in place. The `Box` allocates on the heap, so the future has a stable address. You can safely hold a pointer to it (e.g. for self-referential futures). The `Box` itself can be moved, but the heap allocation's address is stable.

</details>


### Q554. Can you `mem::swap` two `Pin<Box<T>>` values?

- A) Yes; swapping is always allowed
- B) No; it would move the pinned data
- C) Yes; only if `T: Unpin`
- D) No; `Pin` prevents all operations

<details>
<summary>Show Answer</summary>

**Q554. C)** Swapping would move the inner `T` values. If `T: Unpin`, moving is safe, so swap is allowed. If `T` is not `Unpin` (e.g. self-referential), swapping would invalidate internal pointers, so it's forbidden.

</details>


### Q555. What is `Pin::new_unchecked`?

- A) A safe constructor for `Pin`
- B) An `unsafe` constructor; you must guarantee the pointee will not be moved
- C) A way to unpin a value
- D) A deprecated function

<details>
<summary>Show Answer</summary>

**Q555. B)** `Pin::new_unchecked` is `unsafe`. You must uphold the pinning invariant: the pointee must not be moved until it's dropped. Used when implementing custom pinned types (e.g. futures) where you can guarantee this.

</details>


### Q556. Why do most async futures implement `Unpin`?

- A) They don't; most don't implement `Unpin`
- B) Because they're generated by the compiler and don't have self-references
- C) To allow them to be moved between tasks
- D) For compatibility with `Pin`

<details>
<summary>Show Answer</summary>

**Q556. B)** Simple async blocks that don't hold references into themselves are `Unpin`. The compiler generates a future struct; if it has no self-references, it's `Unpin`. More complex futures (e.g. with `select!` or manual impl) may be `!Unpin`.

</details>


### Q557. `impl !Unpin for MyType` — what does `!Unpin` mean?

- A) MyType does not implement Unpin (opt-out)
- B) MyType explicitly implements Unpin
- C) MyType cannot be pinned
- D) Syntax error

<details>
<summary>Show Answer</summary>

**Q557. A)** `!Unpin` is an opt-out. By default, types are `Unpin`. `impl !Unpin for MyType` explicitly marks `MyType` as not implementing `Unpin`, so it cannot be safely moved when pinned.

</details>


### Q558. What does `Pin::as_mut` do?

- A) Converts `Pin<&mut T>` to `&mut T`
- B) Converts `Pin<P>` to `Pin<&mut T>` where `P: DerefMut`, for method chaining
- C) Unpins the value
- D) Creates a mutable reference to the `Pin`

<details>
<summary>Show Answer</summary>

**Q558. B)** `Pin::as_mut` takes `Pin<&mut P>` (where `P: DerefMut`) and returns `Pin<&mut P::Target>`. It's used to get a pinned projection for method chains, e.g. when polling a future.

</details>


### Q559. When would you use `Box::pin(x)`?

- A) To allocate `x` on the heap
- B) To create a `Pin<Box<T>>` — heap-allocate and pin in one step
- C) To prevent `x` from being dropped
- D) To create a pinned reference

<details>
<summary>Show Answer</summary>

**Q559. B)** `Box::pin(x)` is equivalent to `Pin::from(Box::new(x))`. It allocates on the heap and pins the value. Convenient for creating pinned futures or other self-referential types.

</details>


### Q560. A future that holds `&mut self` in its poll state — is it `Unpin`?

- A) Yes; all futures are Unpin
- B) No; it's self-referential (the future holds a reference into its own state)
- C) Yes; references are Unpin
- D) Only if the reference is `'static`

<details>
<summary>Show Answer</summary>

**Q560. B)** If the future holds a reference that points into its own data (e.g. a field), it's self-referential. Moving the future would invalidate that reference. Such futures are `!Unpin` and must be pinned before polling.

</details>


### Q561. What does `#[repr(C)]` do?

- A) Makes the type `Copy`
- B) Uses C-compatible layout (field order, alignment, padding)
- C) Enables C interop only
- D) Packs the struct with no padding

<details>
<summary>Show Answer</summary>

**Q561. B)** `#[repr(C)]` uses the same layout rules as C: fields in declaration order, platform-specific alignment and padding. Used for FFI and when you need a predictable layout.

</details>


### Q562. `#[repr(packed)]` — what is the trade-off?

- A) Smaller size, potentially unaligned access (slower or UB on some platforms)
- B) Faster access, larger size
- C) No effect in Rust
- D) Guarantees no padding

<details>
<summary>Show Answer</summary>

**Q562. A)** `packed` removes padding, minimizing size. Fields may be unaligned. Unaligned loads can be slower or cause UB on strict architectures. Use with care, especially for types with alignment requirements.

</details>


### Q563. `size_of::<Option<&i32>>()` — what is it?

- A) `size_of::<&i32>() + 1` (pointer + discriminant)
- B) Same as `size_of::<&i32>()` (null pointer optimization)
- C) Twice the size of a pointer
- D) Platform-dependent

<details>
<summary>Show Answer</summary>

**Q563. B)** Rust applies the null pointer optimization: `Option<&T>` has the same size as `&T`. `None` is represented as the null pointer. So `Option<&i32>` is one pointer size.

</details>


### Q564. `align_of::<u64>()` on a 64-bit system?

- A) 1
- B) 4
- C) 8
- D) 16

<details>
<summary>Show Answer</summary>

**Q564. C)** `u64` is 8 bytes and typically has 8-byte alignment on 64-bit systems. `align_of` returns the alignment requirement.

</details>


### Q565. What is a ZST (Zero-Sized Type)?

- A) A type with size 0 and alignment 0
- B) A type like `()` or `PhantomData<T>` that has size 0
- C) An empty struct
- D) A type that cannot be instantiated

<details>
<summary>Show Answer</summary>

**Q565. B)** ZSTs have `size_of == 0`. Examples: `()`, `PhantomData<T>`, `std::marker::PhantomPinned`. They take no memory. Arrays of ZSTs can have arbitrary length with no size change.

</details>


### Q566. `size_of::<[u8; 0]>()`?

- A) 0
- B) 1
- C) Undefined
- D) Platform-dependent

<details>
<summary>Show Answer</summary>

**Q566. A)** A zero-length array is a ZST. `size_of::<[u8; 0]>() == 0`. No elements, no size.

</details>


### Q567. `#[repr(transparent)]` on a struct with one field — what does it guarantee?

- A) The struct has the same layout as its single non-ZST field
- B) The struct is a ZST
- C) The struct has no padding
- D) The struct can be transmuted to the field type

<details>
<summary>Show Answer</summary>

**Q567. A)** `#[repr(transparent)]` means the struct has the same layout and ABI as its single non-zero-sized field. Used for newtype wrappers (e.g. `struct Meters(f64)`) that must be ABI-compatible with the inner type.

</details>


### Q568. Why might a struct have padding between fields?

- A) To optimize cache performance
- B) To satisfy alignment requirements of subsequent fields
- C) Rust never adds padding
- D) To make the struct larger

<details>
<summary>Show Answer</summary>

**Q568. B)** Padding is inserted to align fields. E.g. `struct S { a: u8, b: u64 }` — `b` needs 8-byte alignment, so padding is added after `a`. This avoids unaligned access.

</details>


### Q569. `std::mem::size_of_val(&x)` vs `size_of::<T>()` where `x: T`?

- A) They're always equal
- B) `size_of_val` handles DSTs (e.g. slices, trait objects) by including the length/vtable
- C) `size_of_val` is smaller for large types
- D) `size_of_val` doesn't exist

<details>
<summary>Show Answer</summary>

**Q569. B)** `size_of::<T>()` requires `T: Sized`. `size_of_val` works with references to DSTs: for `&[T]` it includes the slice length; for `&dyn Trait` it includes the vtable. It gives the runtime size.

</details>


### Q570. `#[repr(u8)] enum E { A, B }` — what is the size of `E`?

- A) 0 (it's an enum)
- B) 1 byte (discriminant)
- C) 2 bytes
- D) Same as the largest variant

<details>
<summary>Show Answer</summary>

**Q570. B)** `#[repr(u8)]` makes the discriminant a `u8`. With no data in the variants, the enum is 1 byte. The repr controls the discriminant size and layout.

</details>


### Q571. In what order are struct fields dropped?

- A) Reverse declaration order
- B) Declaration order
- C) Arbitrary
- D) Reverse declaration order (last declared dropped first)

<details>
<summary>Show Answer</summary>

**Q571. D)** Fields are dropped in reverse declaration order: the last field is dropped first, then the second-to-last, etc. This matches how they're constructed.

</details>


### Q572. `mem::forget(x)` — what happens to `x`?

- A) `x` is dropped immediately
- B) `x` is never dropped; its destructor is not run (intentional leak)
- C) `x` is moved and then dropped
- D) Undefined behavior

<details>
<summary>Show Answer</summary>

**Q572. B)** `mem::forget` consumes `x` and never runs its destructor. The memory/resources are leaked. Used when you've transferred ownership to FFI or when you intentionally want to leak (e.g. for `'static` data).

</details>


### Q573. What is `ManuallyDrop<T>` for?

- A) To prevent `T` from being dropped when the `ManuallyDrop` goes out of scope
- B) To manually trigger the drop
- C) To delay the drop
- D) To make `T` `Copy`

<details>
<summary>Show Answer</summary>

**Q573. A)** `ManuallyDrop` wraps `T` and does not run `T`'s destructor when it's dropped. You can use `ManuallyDrop::into_inner` to take ownership and drop it yourself, or use `mem::forget` to never drop.

</details>


### Q574. `mem::replace(&mut dest, src)` — what does it return?

- A) `src`
- B) The old value of `dest`
- C) `(dest, src)`
- D) Nothing; it returns `()`

<details>
<summary>Show Answer</summary>

**Q574. B)** `mem::replace` writes `src` into `dest` and returns the previous value of `dest`. Useful when you need to take a value out of a mutable reference and leave something in its place.

</details>


### Q575. When is `Drop::drop` called?

- A) When the value goes out of scope
- B) Automatically by the compiler when the value is no longer needed
- C) You must call it explicitly
- D) When `drop()` is called

<details>
<summary>Show Answer</summary>

**Q575. B)** The compiler inserts drop glue that runs `Drop::drop` when the value goes out of scope. You never call `Drop::drop` directly (it's `&mut self` and would require special handling). You use `drop(value)` to run the destructor early.

</details>


### Q576. Can you call `drop` on a value twice?

- A) Yes; `drop` borrows
- B) No; `drop` consumes the value
- C) Yes; with `Copy` types
- D) Only with `Rc`

<details>
<summary>Show Answer</summary>

**Q576. B)** `drop` takes ownership. After `drop(x)`, `x` is moved and cannot be used again. Calling `drop(x)` twice would be a use-after-move error.

</details>


### Q577. In a tuple `(a, b)`, which is dropped first?

- A) `a`
- B) `b`
- C) Simultaneously
- D) Depends on drop order

<details>
<summary>Show Answer</summary>

**Q577. B)** Tuples drop in reverse order: `b` (last field) is dropped first, then `a`. Same as struct fields.

</details>


### Q578. `mem::take(&mut x)` where `x: Option<T>` — what does it do?

- A) Returns `x` and leaves `x` as `None`
- B) Returns `x` and leaves `x` unchanged
- C) Replaces `x` with `None` and returns the old value
- D) Same as `x.take()`

<details>
<summary>Show Answer</summary>

**Q578. D)** `mem::take` is equivalent to `mem::replace(x, Default::default())`. For `Option`, that's `replace(x, None)`, returning the old value. Same as `Option::take`.

</details>


### Q579. Why might you use `ManuallyDrop` in unsafe code?

- A) To avoid double-free when transferring ownership to FFI
- B) To make types `Send`
- C) To optimize drop order
- D) To prevent leaks

<details>
<summary>Show Answer</summary>

**Q579. A)** When passing ownership to FFI, the foreign code will eventually free the memory. If Rust also ran the destructor, that would be double-free. `ManuallyDrop` prevents Rust from dropping, so you transfer ownership without double-free.

</details>


### Q580. `let x = vec![1,2,3]; drop(x); println!("{:?}", x);` — what happens?

- A) Prints the vec
- B) Compile error: use of moved value
- C) Prints nothing
- D) Undefined behavior

<details>
<summary>Show Answer</summary>

**Q580. B)** `drop(x)` consumes `x`. After the call, `x` is no longer valid. Using `x` in `println!` is a use-after-move, which the compiler rejects.

</details>


### Q581. What does `Cow<'a, T>` stand for?

- A) Copy on Write
- B) Clone on Write
- C) Copy or Write
- D) Clone or Write

<details>
<summary>Show Answer</summary>

**Q581. B)** `Cow` = Clone on Write. It holds either a borrowed value (`Borrowed`) or an owned value (`Owned`). When you need to mutate, it clones the borrowed data and switches to owned.

</details>


### Q582. When would you use `Cow<str>`?

- A) When you always own the string
- B) When a function might return either a borrowed slice or an owned String
- C) When you need a mutable string
- D) For string literals only

<details>
<summary>Show Answer</summary>

**Q582. B)** `Cow<str>` can be `Cow::Borrowed(&str)` or `Cow::Owned(String)`. Functions that sometimes return a slice (e.g. from a cache) and sometimes an owned string (when they need to allocate) use `Cow` to avoid unnecessary allocation.

</details>


### Q583. `Copy` vs `Clone`: what's the difference?

- A) `Copy` is implicit; `Clone` requires explicit `.clone()`
- B) `Copy` implies `Clone`; `Copy` types are copied implicitly on assignment
- C) `Copy` is for small types only
- D) `Clone` is a superset of `Copy`

<details>
<summary>Show Answer</summary>

**Q583. B)** `Copy` is a marker trait. `Copy` types are implicitly duplicated (bitwise copy) on assignment and when passed by value. No `clone()` needed. `Clone` requires explicit `.clone()`. `Copy: Clone`.

</details>


### Q584. Why doesn't `String` implement `Copy`?

- A) It's too large
- B) Copying would duplicate the heap allocation; both would try to free it (double-free)
- C) It's not `Clone`
- D) It's a pointer

<details>
<summary>Show Answer</summary>

**Q584. B)** `Copy` means implicit bitwise copy. `String` contains a pointer to heap data. Implicit copy would create two `String`s pointing to the same buffer. When both are dropped, both would free it — double-free, undefined behavior. So `String` is `Clone` but not `Copy`.

</details>


### Q585. `let x = 5; let y = x;` — does `x` remain valid?

- A) No; `x` is moved
- B) Yes; `i32` is `Copy`, so `y` gets a copy
- C) Only if we use `clone`
- D) No; integers are special

<details>
<summary>Show Answer</summary>

**Q585. B)** `i32` is `Copy`. `let y = x` copies the value. Both `x` and `y` are valid. No move occurs.

</details>


### Q586. `Cow::to_mut(&mut cow)` — when does it allocate?

- A) Always
- B) Only when the `Cow` is `Borrowed` (converts to owned)
- C) Never
- D) When the borrowed value is large

<details>
<summary>Show Answer</summary>

**Q586. B)** `to_mut` returns `&mut T`. If the `Cow` is `Borrowed`, it must clone the data to get owned mutable storage, then returns a reference to that. If already `Owned`, it just returns `&mut` to the existing data. Allocates only when `Borrowed`.

</details>


### Q587. Can a type implement `Copy` without `Clone`?

- A) Yes
- B) No; `Copy` requires `Clone` (Copy: Clone)
- C) Only for ZSTs
- D) Only with `unsafe`

<details>
<summary>Show Answer</summary>

**Q587. B)** `Copy` is a subtrait of `Clone`: `pub trait Copy: Clone`. So any `Copy` type must also implement `Clone`. You cannot implement `Copy` without `Clone`.

</details>


### Q588. `impl Copy for MyStruct` — what must `MyStruct`'s fields be?

- A) All `Copy`
- B) All `Clone`
- C) All `Sized`
- D) No special requirement

<details>
<summary>Show Answer</summary>

**Q588. A)** For a struct to be `Copy`, every field must be `Copy`. The compiler enforces this. If any field is not `Copy`, you cannot derive or implement `Copy` for the struct.

</details>


### Q589. `fn f(x: Vec<i32>)` and you call `f(v)` — what happens to `v`?

- A) `v` is copied; both caller and callee have a copy
- B) `v` is moved; the caller cannot use `v` after
- C) `v` is borrowed
- D) `v` is cloned

<details>
<summary>Show Answer</summary>

**Q589. B)** `Vec` is not `Copy`. Passing `v` to `f` moves it. Ownership transfers to `f`. The caller cannot use `v` afterward.

</details>


### Q590. `impl Clone for MyType` — what does `clone` typically do?

- A) Bitwise copy
- B) Create a logically independent copy (may allocate, duplicate resources)
- C) Return a reference
- D) Move the value

<details>
<summary>Show Answer</summary>

**Q590. B)** `Clone::clone` should produce an independent copy. For `String`, it allocates a new buffer and copies the bytes. For `Vec`, it allocates and copies elements. The result is a new value that can be dropped independently.

</details>


### Q591. Where does `Box::new(x)` allocate?

- A) On the stack
- B) On the heap (using the global allocator)
- C) In static memory
- D) Depends on `x`'s size

<details>
<summary>Show Answer</summary>

**Q591. B)** `Box` allocates on the heap via the global allocator. The `Box` itself (the pointer) is on the stack; the pointee is on the heap.

</details>


### Q592. What is `std::alloc::Global`?

- A) A thread-local allocator
- B) The default global heap allocator
- C) A no-op allocator
- D) An allocator for static memory

<details>
<summary>Show Answer</summary>

**Q592. B)** `Global` is the default allocator used by `Box`, `Vec`, etc. It's the process's global heap allocator.

</details>


### Q593. Can you have a memory leak in safe Rust?

- A) No; Rust prevents all leaks
- B) Yes; e.g. `Rc` cycles, `Box::leak`, `mem::forget`
- C) Only with `unsafe`
- D) Only with `RefCell`

<details>
<summary>Show Answer</summary>

**Q593. B)** Safe Rust can leak: `Rc` cycles, `Box::leak`, `mem::forget`, or creating an infinite `Vec` in a global. Rust guarantees memory safety (no use-after-free, no data races) but not absence of leaks.

</details>


### Q594. `Box::leak(Box::new(42))` returns `&'static mut i32`. What happens to the allocation?

- A) Freed when the reference goes out of scope
- B) Freed when the program ends
- C) Never freed; intentionally leaked
- D) Freed by the caller

<details>
<summary>Show Answer</summary>

**Q594. C)** `Box::leak` consumes the `Box` and never runs its destructor. The allocation is intentionally leaked. The returned reference is valid for `'static` (rest of program) but the memory is never freed.

</details>


### Q595. Stack vs heap: when is heap allocation necessary?

- A) Always for large data
- B) When the size is unknown at compile time, or when you need to outlive the current frame
- C) Never; stack is always sufficient
- D) Only for `Vec`

<details>
<summary>Show Answer</summary>

**Q595. B)** Heap is needed when: size is dynamic (e.g. `Vec`), data must outlive the current stack frame, or you need shared ownership. Stack is for fixed-size, local data.

</details>


### Q596. What does `alloc::alloc::alloc(layout)` return?

- A) A `Box` to the allocated memory
- B) A `*mut u8` (raw pointer) to the allocated block, or null on failure
- C) A `Vec`
- D) A `Result`

<details>
<summary>Show Answer</summary>

**Q596. B)** The low-level allocator returns a raw pointer `*mut u8` to the allocated memory, or null if allocation fails. You must manually deallocate with `dealloc` and uphold safety invariants.

</details>


### Q597. `#[global_allocator]` — what is it for?

- A) To use a custom allocator for a single crate
- B) To replace the global heap allocator for the entire program
- C) To create a thread-local allocator
- D) To disable allocation

<details>
<summary>Show Answer</summary>

**Q597. B)** `#[global_allocator]` designates a type that implements `GlobalAlloc` as the program's global allocator. All `Box`, `Vec`, etc. use it. Used for custom allocators (jemalloc, tracking allocators, etc.).

</details>


### Q598. Is a `Vec` that grows without bound in a loop a memory leak?

- A) No; it's just large
- B) Yes; if the Vec is never used or freed, it's a leak
- C) Only if it's in a global
- D) Vec cannot leak

<details>
<summary>Show Answer</summary>

**Q598. B)** If you push to a `Vec` in a loop and never use or drop it (e.g. it's stored in a cycle or forgotten), the allocation is leaked. The memory is allocated but never freed. That's a leak.

</details>


### Q599. `Box::into_raw(Box::new(x))` — what do you get?

- A) A `Box` that will not be dropped
- B) A `*mut T`; you must manually drop and deallocate
- C) A leaked `Box`
- D) Ownership of `x`

<details>
<summary>Show Answer</summary>

**Q599. B)** `into_raw` consumes the `Box` and returns `*mut T`. The allocation is not freed. You must call `from_raw` to get a `Box` back and drop it, or manually deallocate. Used for FFI or manual memory management.

</details>


### Q600. Why might you use `Box::leak` in practice?

- A) To avoid double-free
- B) To create `'static` data at runtime (e.g. for lazy_static-style init, or passing to APIs requiring `'static`)
- C) To speed up allocation
- D) To prevent use-after-free

<details>
<summary>Show Answer</summary>

**Q600. B)** `Box::leak` gives you `&'static mut T`. Useful when you need a `'static` reference but the value is created at runtime—e.g. initializing a global, or satisfying APIs that require `'static` (e.g. spawning threads, storing in `lazy_static`).

</details>
